# Gold Layer — Taxi Star Schema

Dimensional model for the taxi booking dataset. Reads from `silver_taxi_data` in the `team-1-taxi-service` pipeline.  
Builds `fact_booking` plus seven dimensions: `dim_date`, `dim_driver`, `dim_vehicle`, `dim_zone`, `dim_booking_source`, `dim_payment_type`, `dim_capability`.

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql import Row

In [0]:
# ---------------------------------------------------------------------------
# dim_date — Date dimension
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_date",
    comment="Date dimension derived from pickup_due and completed timestamps"
)
def dim_date():
    """
    Conformed date dimension covering every calendar date in the dataset.
    Dates sourced from pickup_due and completed timestamps in silver.
    Surrogate key: abs(xxhash64(date_string)) — idempotent across re-runs.
    Includes a -1 / Unknown member for late-arriving or missing dates.
    """
    silver = dlt.read("silver_taxi_data")

    # --- Extract distinct dates from both timestamp columns ---
    dates = (
        silver.select(F.to_date("pickup_due").alias("date"))
        .union(silver.select(F.to_date("completed").alias("date")))
        .filter(F.col("date").isNotNull())
        .distinct()
    )

    # --- Build calendar attributes ---
    dates = dates.select(
        F.abs(F.xxhash64(F.col("date").cast("string"))).alias("date_key"),
        F.col("date"),
        F.dayofweek("date").alias("day_of_week"),
        F.date_format("date", "EEEE").alias("day_name"),
        F.dayofmonth("date").alias("day_of_month"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.quarter("date").alias("quarter"),
        F.year("date").alias("year"),
        F.dayofweek("date").isin(1, 7).cast("boolean").alias("is_weekend"),
    )

    # --- Append -1 / Unknown default member ---
    unknown = spark.sql("""
        SELECT
            CAST(-1 AS BIGINT)    AS date_key,
            CAST(NULL AS DATE)    AS date,
            CAST(-1 AS INT)       AS day_of_week,
            'Unknown'             AS day_name,
            CAST(-1 AS INT)       AS day_of_month,
            CAST(-1 AS INT)       AS month,
            'Unknown'             AS month_name,
            CAST(-1 AS INT)       AS quarter,
            CAST(-1 AS INT)       AS year,
            CAST(NULL AS BOOLEAN) AS is_weekend
    """)

    return dates.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# dim_driver — Driver dimension
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_driver",
    comment="Driver dimension keyed on driver ID"
)
def dim_driver():
    """
    One row per distinct driver. Surrogate key: abs(xxhash64(driver_id)).
    Includes a -1 / Unknown member.
    """
    silver = dlt.read("silver_taxi_data")

    drivers = (
        silver.select(F.col("driver").alias("driver_id"))
        .filter(F.col("driver_id").isNotNull())
        .distinct()
        .select(
            F.abs(F.xxhash64(F.col("driver_id").cast("string"))).alias("driver_key"),
            F.col("driver_id"),
        )
    )

    unknown = spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS driver_key, CAST(-1 AS INT) AS driver_id
    """)

    return drivers.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# dim_vehicle — Vehicle dimension
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_vehicle",
    comment="Vehicle dimension keyed on vehicle ID"
)
def dim_vehicle():
    """
    One row per distinct vehicle. Surrogate key: abs(xxhash64(vehicle_id)).
    Includes a -1 / Unknown member.
    """
    silver = dlt.read("silver_taxi_data")

    vehicles = (
        silver.select(F.col("vehicle").alias("vehicle_id"))
        .filter(F.col("vehicle_id").isNotNull())
        .distinct()
        .select(
            F.abs(F.xxhash64(F.col("vehicle_id").cast("string"))).alias("vehicle_key"),
            F.col("vehicle_id"),
        )
    )

    unknown = spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS vehicle_key, CAST(-1 AS INT) AS vehicle_id
    """)

    return vehicles.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# dim_zone — Conformed zone dimension (pickup + destination)
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_zone",
    comment="Conformed zone dimension — union of pickup and destination zones, joined twice in fact"
)
def dim_zone():
    """
    Conformed dimension: pickup_zone and destination_zone share the same
    vocabulary, so one dim serves both roles. Fact joins this table twice
    (pickup_zone_key, destination_zone_key).
    Surrogate key: abs(xxhash64(zone_name)). Includes -1 / Unknown.
    """
    silver = dlt.read("silver_taxi_data")

    zones = (
        silver.select(F.col("pickup_zone").alias("zone_name"))
        .union(silver.select(F.col("destination_zone").alias("zone_name")))
        .filter(F.col("zone_name").isNotNull())
        .distinct()
        .select(
            F.abs(F.xxhash64("zone_name")).alias("zone_key"),
            F.col("zone_name"),
        )
    )

    unknown = spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS zone_key, 'Unknown' AS zone_name
    """)

    return zones.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# dim_booking_source — Booking source dimension
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_booking_source",
    comment="Booking source dimension (how the booking was placed)"
)
def dim_booking_source():
    """
    One row per distinct booking source.
    Surrogate key: abs(xxhash64(booking_source)). Includes -1 / Unknown.
    """
    silver = dlt.read("silver_taxi_data")

    sources = (
        silver.select("booking_source")
        .filter(F.col("booking_source").isNotNull())
        .distinct()
        .select(
            F.abs(F.xxhash64("booking_source")).alias("booking_source_key"),
            F.col("booking_source"),
        )
    )

    unknown = spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS booking_source_key, 'Unknown' AS booking_source
    """)

    return sources.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# dim_payment_type — Payment type dimension
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_payment_type",
    comment="Payment type dimension (how the trip was paid)"
)
def dim_payment_type():
    """
    One row per distinct payment type.
    Surrogate key: abs(xxhash64(payment_type)). Includes -1 / Unknown.
    """
    silver = dlt.read("silver_taxi_data")

    types = (
        silver.select("payment_type")
        .filter(F.col("payment_type").isNotNull())
        .distinct()
        .select(
            F.abs(F.xxhash64("payment_type")).alias("payment_type_key"),
            F.col("payment_type"),
        )
    )

    unknown = spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS payment_type_key, 'Unknown' AS payment_type
    """)

    return types.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# dim_capability — Junk dimension (capability boolean flags)
# ---------------------------------------------------------------------------
@dlt.table(
    name="dim_capability",
    comment="Junk dimension — boolean flags parsed from capability code combinations"
)
def dim_capability():
    """
    Junk dimension: each distinct capability combination becomes one row
    with individual boolean flags. ~82% of Silver rows have NULL capabilities,
    handled by the -1 / Unknown member. Not exploded to a bridge table.

    Capability codes: Z=Card Reader, D=Delivery, H=High Car, L=Low Car,
    W=Wheelchair, M=Minibus, F=Female Driver, V=VIP, T=Tour, P=Pet,
    6/7/8=Seater.
    """
    silver = dlt.read("silver_taxi_data")

    # --- Distinct non-NULL capability strings (already readable from Silver UDF) ---
    caps = (
        silver.select("capabilities")
        .filter(F.col("capabilities").isNotNull())
        .distinct()
    )

    # --- Parse readable names into boolean flags ---
    caps = caps.select(
        F.abs(F.xxhash64("capabilities")).alias("capability_key"),
        F.col("capabilities"),
        F.col("capabilities").contains("Card Reader").cast("boolean").alias("has_card_reader"),
        F.col("capabilities").contains("Delivery").cast("boolean").alias("has_delivery"),
        F.col("capabilities").contains("High Car").cast("boolean").alias("has_high_car"),
        F.col("capabilities").contains("Low Car").cast("boolean").alias("has_low_car"),
        F.col("capabilities").contains("Wheelchair").cast("boolean").alias("has_wheelchair"),
        F.col("capabilities").contains("Minibus").cast("boolean").alias("has_minibus"),
        F.col("capabilities").contains("Female").cast("boolean").alias("has_female_driver"),
        F.col("capabilities").contains("VIP").cast("boolean").alias("has_vip"),
        F.col("capabilities").contains("Tour").cast("boolean").alias("has_tour"),
        F.col("capabilities").contains("Pet").cast("boolean").alias("has_pet"),
        F.col("capabilities").contains("6 seater").cast("boolean").alias("is_6_seater"),
        F.col("capabilities").contains("7 seater").cast("boolean").alias("is_7_seater"),
        F.col("capabilities").contains("8 seater").cast("boolean").alias("is_8_seater"),
    )

    # --- Append -1 / Unknown default member (all flags false) ---
    unknown = spark.sql("""
        SELECT
            CAST(-1 AS BIGINT)    AS capability_key,
            CAST(NULL AS STRING)  AS capabilities,
            false AS has_card_reader, false AS has_delivery,
            false AS has_high_car,    false AS has_low_car,
            false AS has_wheelchair,   false AS has_minibus,
            false AS has_female_driver,false AS has_vip,
            false AS has_tour,         false AS has_pet,
            false AS is_6_seater,      false AS is_7_seater,
            false AS is_8_seater
    """)

    return caps.unionByName(unknown)

In [0]:
# ---------------------------------------------------------------------------
# fact_booking — Central fact table (grain: one row per taxi booking)
# ---------------------------------------------------------------------------
_PRICE_PER_MILE_UPPER_FENCE = 8.16  # Q3 + 3*IQR threshold from Silver analysis


@dlt.table(
    name="fact_booking",
    comment="Fact table at booking grain — one row per taxi booking with surrogate dimension keys and additive measures"
)
def fact_booking():
    """
    Central fact table. Grain: one booking_id = one row (771,150 rows, verified unique).

    Surrogate FK keys reference all seven dimensions via LEFT JOIN + coalesce(-1).
    Derived measures (trip_duration_minutes, wait_time_minutes, total_time_taken,
    price_per_mile) and quality flags (is_completed, is_valid_distance, etc.)
    are computed here in Gold rather than Silver.

    ~7,206 rows expected with destination_zone_key = -1 (known missing-destination
    issue, not a broken join).
    """
    silver = dlt.read("silver_taxi_data")

    # --- Dimension lookups (aliased to avoid column ambiguity) ---
    lkp_date = dlt.read("dim_date").select(
        F.col("date_key").alias("_dk"), F.col("date").alias("_d")
    )
    lkp_driver = dlt.read("dim_driver").select(
        F.col("driver_key").alias("_drk"), F.col("driver_id").alias("_dri")
    )
    lkp_vehicle = dlt.read("dim_vehicle").select(
        F.col("vehicle_key").alias("_vk"), F.col("vehicle_id").alias("_vi")
    )
    lkp_pzone = dlt.read("dim_zone").select(
        F.col("zone_key").alias("_pzk"), F.col("zone_name").alias("_pzn")
    )
    lkp_dzone = dlt.read("dim_zone").select(
        F.col("zone_key").alias("_dzk"), F.col("zone_name").alias("_dzn")
    )
    lkp_bsource = dlt.read("dim_booking_source").select(
        F.col("booking_source_key").alias("_bsk"),
        F.col("booking_source").alias("_bsn"),
    )
    lkp_ptype = dlt.read("dim_payment_type").select(
        F.col("payment_type_key").alias("_ptk"),
        F.col("payment_type").alias("_ptn"),
    )
    lkp_cap = dlt.read("dim_capability").select(
        F.col("capability_key").alias("_ck"),
        F.col("capabilities").alias("_cn"),
    )

    # --- LEFT JOIN silver to all dimensions on natural keys ---
    fact = (
        silver
        .join(lkp_date,    F.to_date(silver["pickup_due"]) == lkp_date["_d"],     "left")
        .join(lkp_driver,  silver["driver"]               == lkp_driver["_dri"],  "left")
        .join(lkp_vehicle, silver["vehicle"]              == lkp_vehicle["_vi"],  "left")
        .join(lkp_pzone,   silver["pickup_zone"]          == lkp_pzone["_pzn"],   "left")
        .join(lkp_dzone,   silver["destination_zone"]     == lkp_dzone["_dzn"],   "left")
        .join(lkp_bsource, silver["booking_source"]       == lkp_bsource["_bsn"], "left")
        .join(lkp_ptype,   silver["payment_type"]         == lkp_ptype["_ptn"],   "left")
        .join(lkp_cap,     silver["capabilities"]         == lkp_cap["_cn"],      "left")
    )

    # --- Select final columns with coalesce to -1 for missing dims ---
    return fact.select(
        # Grain key
        silver["booking_id"],

        # Surrogate FK keys (coalesce NULL -> -1)
        F.coalesce(F.col("_dk"),  F.lit(-1)).cast("bigint").alias("date_key"),
        F.coalesce(F.col("_drk"), F.lit(-1)).cast("bigint").alias("driver_key"),
        F.coalesce(F.col("_vk"),  F.lit(-1)).cast("bigint").alias("vehicle_key"),
        F.coalesce(F.col("_pzk"), F.lit(-1)).cast("bigint").alias("pickup_zone_key"),
        F.coalesce(F.col("_dzk"), F.lit(-1)).cast("bigint").alias("destination_zone_key"),
        F.coalesce(F.col("_bsk"), F.lit(-1)).cast("bigint").alias("booking_source_key"),
        F.coalesce(F.col("_ptk"), F.lit(-1)).cast("bigint").alias("payment_type_key"),
        F.coalesce(F.col("_ck"),  F.lit(-1)).cast("bigint").alias("capability_key"),

        # Additive measures
        silver["price"],
        silver["distance"],

        # Derived measures (moved from Silver -> Gold)
        F.round(
            (F.unix_timestamp(silver["completed"]) - F.unix_timestamp(silver["pickup_due"])) / 60, 2
        ).alias("trip_duration_minutes"),
        F.round(
            (F.unix_timestamp(silver["completed"]) - F.unix_timestamp(silver["time_dispatched"])) / 60, 2
        ).alias("total_time_taken"),
        F.round(
            (F.unix_timestamp(silver["time_picked_up"]) - F.unix_timestamp(silver["time_vehicle_arrived"])) / 60, 2
        ).alias("wait_time_minutes"),
        F.when(
            silver["distance"] > 0,
            F.round(silver["price"] / silver["distance"], 2)
        ).alias("price_per_mile"),

        # Quality flags (moved from Silver -> Gold)
        (silver["trip_status"] == "Completed").cast("boolean").alias("is_completed"),
        (silver["distance"] > 0).cast("boolean").alias("is_valid_distance"),
        (
            silver["time_vehicle_arrived"].isNotNull()
            & silver["time_picked_up"].isNotNull()
        ).cast("boolean").alias("is_valid_timing"),
        (
            (silver["trip_status"] != "Completed")
            | (
                (F.unix_timestamp(silver["completed"]) - F.unix_timestamp(silver["pickup_due"])) >= 0
            )
        ).cast("boolean").alias("is_valid_duration"),
        F.when(
            (silver["distance"] > 0)
            & ((silver["price"] / silver["distance"]) > _PRICE_PER_MILE_UPPER_FENCE),
            F.lit(True)
        ).otherwise(F.lit(False)).alias("is_price_outlier"),

        # Degenerate dimensions
        silver["trip_status"],
        silver["priority"],
        silver["booked_by"],

        # Timestamps (for drill-down)
        silver["pickup_due"],
        silver["completed"],
        silver["time_dispatched"],
        silver["time_vehicle_arrived"],
        silver["time_picked_up"],

        # Coordinates (for geospatial analysis)
        silver["pickup_latitude"],
        silver["pickup_longitude"],
        silver["destination_latitude"],
        silver["destination_longitude"],
    )